# Macro Economics Indicators - Processed

**Autor:** Frederick Salazar Sanchez <br>
**Date:** March 2026 <br>
**Description:** This notebook unifies all macroeconomics indicators into a single dataset from 1960 to 2023

In [46]:
import pandas as pd

# Load Data

In [47]:
df_gdp                 = pd.read_csv('./data/in/countries_gdp.csv', sep=';')
df_gdp_percapita       = pd.read_csv('./data/in/countries_gdp_percapita.csv', sep=';')
df_gdp_variation       = pd.read_csv('./data/in/countries_gdp_variation.csv', sep=';')
df_exports             = pd.read_csv('./data/in/countries_exports.csv', sep=';')
df_inflation           = pd.read_csv('./data/in/countries_inflation.csv', sep=';')
df_inversion           = pd.read_csv('./data/in/countries_inversion_extranjera.csv', sep=';')
df_desempleo           = pd.read_csv('./data/in/countries_desempleo.csv', sep=';')
df_imports             = pd.read_csv('./data/in/countries_imports.csv', sep=';')
df_reservas            = pd.read_csv('./data/in/countries_reservas_internacionales.csv', sep=';')
df_deuda               = pd.read_csv('./data/in/countries_deuda_externa.csv', sep=';')
df_poblacion           = pd.read_csv('./data/in/countries_poblacion.csv', sep=';')
df_cuenta_corriente    = pd.read_csv('./data/in/countries_cuenta_corriente.csv', sep=';')
df_deuda_publica       = pd.read_csv('./data/in/countries_deuda_publica.csv', sep=';')
df_ingresos_tributarios = pd.read_csv('./data/in/countries_ingresos_tributarios.csv', sep=';')
df_gini                = pd.read_csv('./data/in/countries_gini.csv', sep=';')
df_life                = pd.read_csv('./data/in/life_expectancy_dataset.csv', sep=';', decimal=',')

# Select and Rename Columns

In [48]:
base_cols = ['country_code', 'region_name', 'sub_region_name', 'intermediate_region', 'country_name', 'income_group', 'year']

# Archivos con estructura estándar (sep=';', con base_cols completos)
df_gdp                  = df_gdp[base_cols + ['total_gdp']]
df_gdp_percapita        = df_gdp_percapita[base_cols + ['total_gdp_percapita']]
df_gdp_variation        = df_gdp_variation[base_cols + ['variacion']].rename(columns={'variacion': 'gdp_variation'})
df_exports              = df_exports[base_cols + ['exportacion']].rename(columns={'exportacion': 'exports_of_goods_and_services'})
df_inflation            = df_inflation[base_cols + ['inflation']].rename(columns={'inflation': 'inflation_rate'})
df_inversion            = df_inversion[base_cols + ['inversion_extranjera']].rename(columns={'inversion_extranjera': 'foreign_direct_investment'})
df_desempleo            = df_desempleo[base_cols + ['desempleo']].rename(columns={'desempleo': 'unemployment_rate'})
df_imports              = df_imports[base_cols + ['importacion']].rename(columns={'importacion': 'imports_of_goods_and_services'})
df_reservas             = df_reservas[base_cols + ['reservas_internacionales']].rename(columns={'reservas_internacionales': 'international_reserves'})
df_deuda                = df_deuda[base_cols + ['deuda_externa']].rename(columns={'deuda_externa': 'external_debt'})
df_poblacion            = df_poblacion[base_cols + ['poblacion']]
df_cuenta_corriente     = df_cuenta_corriente[base_cols + ['cuenta_corriente']]
df_deuda_publica        = df_deuda_publica[base_cols + ['deuda_publica']]
df_ingresos_tributarios = df_ingresos_tributarios[base_cols + ['ingresos_tributarios']]
df_gini                 = df_gini[base_cols + ['gini']]

# df_life no tiene income_group ni región — solo extraemos las columnas propias
df_life = df_life[['country_code', 'year', 'life_expectancy_women', 'life_expectancy_men']]

# Merge All DataFrames

In [49]:
# Merge base: todos los archivos con estructura estándar se unen por base_cols
df_unified = (
    df_gdp
    .merge(df_gdp_percapita,        on=base_cols, how='left')
    .merge(df_gdp_variation,        on=base_cols, how='left')
    .merge(df_exports,              on=base_cols, how='left')
    .merge(df_inflation,            on=base_cols, how='left')
    .merge(df_inversion,            on=base_cols, how='left')
    .merge(df_desempleo,            on=base_cols, how='left')
    .merge(df_imports,              on=base_cols, how='left')
    .merge(df_reservas,             on=base_cols, how='left')
    .merge(df_deuda,                on=base_cols, how='left')
    .merge(df_poblacion,            on=base_cols, how='left')
    .merge(df_cuenta_corriente,     on=base_cols, how='left')
    .merge(df_deuda_publica,        on=base_cols, how='left')
    .merge(df_ingresos_tributarios, on=base_cols, how='left')
    .merge(df_gini,                 on=base_cols, how='left')
    # Archivos sin income_group/región se unen solo por país + año
    .merge(df_life,                 on=['country_code', 'year'], how='left')
)

print(f"Filas: {len(df_unified):,} | Columnas: {df_unified.shape[1]}")

Filas: 14,190 | Columnas: 24


# Data Transformation

In [50]:
df_unified['total_gdp_million'] = df_unified['total_gdp'] / 1_000_000

cols_numerical = [
    'total_gdp', 'total_gdp_million', 'gdp_variation', 'total_gdp_percapita',
    'exports_of_goods_and_services', 'imports_of_goods_and_services',
    'inflation_rate', 'foreign_direct_investment', 'unemployment_rate',
    'international_reserves', 'external_debt',
    'poblacion', 'cuenta_corriente', 'deuda_publica', 'ingresos_tributarios', 'gini',
    'life_expectancy_women', 'life_expectancy_men'
]

for col in cols_numerical:
    if col in df_unified.columns:
        df_unified[col] = pd.to_numeric(df_unified[col], errors='coerce')

# Variación nominal del PIB per cápita calculada desde total_gdp_percapita (USD corrientes)
df_unified = df_unified.sort_values(['country_code', 'year'])
df_unified['gdp_percapita_variation'] = (
    df_unified.groupby('country_code')['total_gdp_percapita']
    .pct_change() * 100
)

# % deuda externa respecto al PIB (ambos en USD corrientes)
df_unified['external_debt_pct_gdp'] = (df_unified['external_debt'] / df_unified['total_gdp']) * 100

# Reorder Columns

In [51]:
output_cols = [
    'country_code', 'region_name', 'sub_region_name', 'intermediate_region',
    'country_name', 'income_group', 'year',
    # PIB
    'total_gdp', 'total_gdp_million', 'gdp_variation', 'total_gdp_percapita', 'gdp_percapita_variation',
    # Comercio exterior
    'exports_of_goods_and_services', 'imports_of_goods_and_services', 'cuenta_corriente',
    # Precios e inversión
    'inflation_rate', 'foreign_direct_investment',
    # Mercado laboral
    'unemployment_rate',
    # Sector externo y reservas
    'international_reserves', 'external_debt', 'external_debt_pct_gdp',
    # Fiscal
    'deuda_publica', 'ingresos_tributarios',
    # Demografía y desigualdad
    'poblacion', 'gini',
    # Social
    'life_expectancy_women', 'life_expectancy_men'
]

df_unified = df_unified[output_cols]
df_unified.head(3)

,country_code,region_name,sub_region_name,intermediate_region,country_name,income_group,year,total_gdp,total_gdp_million,gdp_variation,...,unemployment_rate,international_reserves,external_debt,external_debt_pct_gdp,deuda_publica,ingresos_tributarios,poblacion,gini,life_expectancy_women,life_expectancy_men
0,ABW,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,CARIBBEAN,ARUBA,INGRESO ALTO,1960,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,0.0,0.0,54922.0,0.0,67.78,60.58
1,ABW,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,CARIBBEAN,ARUBA,INGRESO ALTO,1961,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,0.0,0.0,55578.0,0.0,68.27,60.88
2,ABW,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,CARIBBEAN,ARUBA,INGRESO ALTO,1962,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,0.0,0.0,56320.0,0.0,68.58,61.02


# Country Name Corrections

In [52]:
country_corrections = {
    'COD': 'REPUBLIC DEMOCRATIC OF CONGO',
    'KOR': 'REPUBLIC OF KOREA',
    'MDA': 'REPUBLIC OF MOLDOVA',
    'PSE': 'STATE OF PALESTINE',
    'TZA': 'UNITED REPUBLIC OF TANZANIA'
}

for code, name in country_corrections.items():
    df_unified.loc[df_unified['country_code'] == code, 'country_name'] = name

# Save to CSV

In [53]:
df_unified.to_csv(
    './data/out/macro_economics_indicators_2026.csv',
    index=False,
    sep=';',
    decimal=',',
    float_format='%.2f'
)

In [54]:
df_unified.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14190 entries, 0 to 14189
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country_code                   14190 non-null  object 
 1   region_name                    14190 non-null  object 
 2   sub_region_name                14190 non-null  object 
 3   intermediate_region            14190 non-null  object 
 4   country_name                   14190 non-null  object 
 5   income_group                   14190 non-null  object 
 6   year                           14190 non-null  int64  
 7   total_gdp                      14190 non-null  float64
 8   total_gdp_million              14190 non-null  float64
 9   gdp_variation                  14190 non-null  float64
 10  total_gdp_percapita            14190 non-null  float64
 11  gdp_percapita_variation        11634 non-null  float64
 12  exports_of_goods_and_services  14190 non-null 